# Module 3: Audience Sentiment & Relatability Classifier

## 1. Problem Formulation & Orthogonal Dimensions

In social media engagement intelligence, audience response must not be treated as a single sentiment polarity axis.
We strictly formulate **two orthogonal analytical dimensions**:

1. **Relatability (Audience Identification):**
   - **Relatable:** Comments expressing shared personal lived experience, vulnerability, belonging, or mutual struggle (*"This happens to me every morning"*, *"Thought I was the only one going through this"*, *"Story of my life right now"*).
   - **Neutral:** Generic praise without identification (*"Great video!"*), gear/technical inquiries (*"What camera did you use?"*), objective critique, or unrelated commentary.
   - **Uncertain / Insufficient Context:** Ambiguous or boundary cases (*"maybe"*, *"👀"*) preserved with explicit diagnostic status.

2. **Sentiment Polarity (Emotional Valence):**
   - Positive, Negative, Neutral, or Mixed.
   - Crucially, **Negative Sentiment comments can be highly Relatable** (e.g. sharing anxiety, burnout, or frustration), while **Positive Sentiment comments can be completely Non-Relatable** (e.g. praising lighting or editing).

> **Dataset Provenance Notice:** This notebook uses a curated synthetic benchmark comments corpus ($N=500$) designed to evaluate pipeline contracts, safety guards, and statistical separation without exposing proprietary social media credentials.

In [ ]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import json
import pandas as pd
import numpy as np

from src.nlp.text_preprocessor import TextPreprocessor
from src.nlp.sentiment_analyzer import SentimentAnalyzer
from src.nlp.relatability_classifier import RelatabilityClassifier
from src.data_ingestion.schemas import ClassificationStatus, RelatabilityLabel, SentimentLabel

print("NLP modules successfully loaded.")

## 2. Ingest and Inspect the Benchmark Corpus

In [ ]:
df_raw = pd.read_csv("../data/sample/sample_comments.csv")
print(f"Total comments: {len(df_raw):,}")
print(f"Unique comment texts: {df_raw['comment_text'].nunique():,}")
display(df_raw.head(8))
print("Ground truth relatability distribution:")
print(df_raw['ground_truth_relatability'].value_counts())

## 3. Text Preprocessing & Language Safety Guards

Our `TextPreprocessor` ensures:
- Preservation of negations (*"not"*, *"never"*, *"cannot"*) and conversational punctuation (*"!"*, *"?"*).
- Rejection of empty/whitespace inputs as `EMPTY_TEXT`.
- Rejection of single characters/pure emojis as `INSUFFICIENT_CONTEXT`.
- Non-English scripts/languages detected and marked as `UNSUPPORTED_LANGUAGE` rather than forcing English models.

In [ ]:
prep = TextPreprocessor()

test_cases = [
    "This happens to me every single morning! 😭",
    "I don't think this advice is helpful at all.",
    "What camera and lens did you use for this?",
    "👀🔥",
    "C'est tellement vrai pour moi aussi dans ma vie.",
    "   "
]

for t in test_cases:
    res = prep.preprocess(t)
    print(f"Raw: '{t}' -> Valid: {res.is_valid} | Status: {res.status.value} | Lang: {res.language}")

## 4. Orthogonal Sentiment Polarity Engine

The `SentimentAnalyzer` uses valence polarity scoring with intensifier boosting and negation flipping.

In [ ]:
sent_analyzer = SentimentAnalyzer()

sent_examples = [
    "Amazing video, loved every single second of this explanation!",
    "This advice was not good at all, completely useless.",
    "My anxiety has been through the roof all week, feeling terrible.",
    "What microphone is that on your desk?"
]

for s in sent_examples:
    res = sent_analyzer.analyze_text(s)
    print(f"Text: '{s[:45]}...' -> Sentiment: {res['sentiment_label']} (Score: {res['polarity']}, Conf: {res['sentiment_confidence']})")

## 5. Model Benchmarking: Zero-Leakage Holdout Evaluation

To evaluate out-of-sample generalization, comments are partitioned on **unique texts** so that identical comment strings never appear in both train and test splits.
We report two benchmark views:
1. **Multi-Post Repeated Instance Benchmark ($N=114$ rows):** Evaluates performance weighted by comment occurrence across simulated posts.
2. **Strictly Non-Redundant Unique Text Benchmark ($N=28$ distinct comments):** Evaluates non-redundant out-of-sample performance.

> **Circularity & Provenance Disclosure:** The benchmark labels were manually authored with predefined labels. Rule guardrails in the classifier share common phrases with the authored benchmark. Therefore, rule precision is circular and artificially elevated; no independent human annotations or inter-rater reliability scores (e.g. Cohen's Kappa) exist.

In [ ]:
with open("../models/relatability_metadata.json", "r") as f:
    meta = json.load(f)

print("=== A. Multi-Post Repeated Instance Benchmark (N=114) ===")
bench = meta["models_benchmarked"]
bench_df = pd.DataFrame([
    {
        "Model": k.replace("_", " ").title(),
        "Accuracy": f"{v['accuracy']:.1%}",
        "Relatable Precision": f"{v['precision_relatable']:.1%}",
        "Relatable Recall": f"{v['recall_relatable']:.1%}",
        "Macro F1": f"{v['f1_macro']:.3f}",
        "Uncertain Flagged": v.get("uncertain_count", 0)
    }
    for k, v in bench.items()
])
display(bench_df)

if "strictly_unique_text_benchmark" in meta:
    print("\n=== B. Non-Redundant Unique Text Benchmark (N=28) ===")
    uniq_bench = meta["strictly_unique_text_benchmark"]
    uniq_df = pd.DataFrame([
        {
            "Model": k.replace("_", " ").title(),
            "Accuracy": f"{v['accuracy']:.1%}",
            "Relatable Precision": f"{v['precision_relatable']:.1%}",
            "Relatable Recall": f"{v['recall_relatable']:.1%}",
            "Macro F1": f"{v['f1_macro']:.3f}",
            "Uncertain Flagged": v.get("uncertain_count", 0)
        }
        for k, v in uniq_bench.items()
    ])
    display(uniq_df)

### Key Empirical Findings:
- **First-Person Heuristic Failure:** Yields substantial false positives on technical gear questions (*"What camera did you use? I want to buy it"*) and subjective critique (*"I disagree with your third point"*).
- **Rule-Based Matcher Limitations:** High precision on known templates but low recall (57.1% on unique test comments) because varied conversational phrasings bypass rigid keyword lists.
- **Hybrid Multi-Tier Pipeline:** Combines domain guardrails with statistical generalization, reaching **89.3% accuracy** and **0.893 Macro F1** on non-redundant unique test comments (and **90.4% accuracy** on the repeated split), while withholding ambiguous borderline cases as `Uncertain`.

## 6. Post-Level Relatability Aggregation

In [ ]:
proc_df = pd.read_parquet("../data/processed/processed_comments.parquet")
classifier = RelatabilityClassifier(model_path="../models/relatability_classifier.joblib")

post_scores = classifier.compute_post_relatability_score(proc_df)
display(post_scores.head(10))

print(f"Average post relatability ratio: {post_scores['relatability_ratio'].mean():.2%}")